In [1]:
!pip -q install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [16]:
import os
import cv2
from ultralytics import YOLO


def draw_box_with_label(frame, x1, y1, x2, y2, label, color=(0, 255, 0), thickness=3):
    font_scale = 1.2
    font_thickness = 3
    padding = 12

    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

    (text_w, text_h), baseline = cv2.getTextSize(
        label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness
    )

    y_text = max(y1, text_h + padding + 5)

    cv2.rectangle(
        frame,
        (x1, y_text - text_h - padding),
        (x1 + text_w + padding, y_text + baseline - 4),
        color,
        -1,
    )

    cv2.putText(
        frame,
        label,
        (x1 + 6, y_text - 6),
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        (0, 0, 0),
        font_thickness,
        cv2.LINE_AA,
    )


def draw_counting_line(frame, y_line, width):
    x_start = 40
    x_end = width - 40

    cv2.line(frame, (x_start, y_line), (x_end, y_line), (255, 80, 255), 16)
    cv2.line(frame, (x_start, y_line), (x_end, y_line), (255, 0, 180), 10)
    cv2.line(frame, (x_start, y_line), (x_end, y_line), (0, 255, 255), 4)

    cv2.circle(frame, (x_start, y_line), 12, (255, 0, 180), -1)
    cv2.circle(frame, (x_end, y_line), 12, (255, 0, 180), -1)
    cv2.circle(frame, (x_start, y_line), 6, (255, 255, 255), -1)
    cv2.circle(frame, (x_end, y_line), 6, (255, 255, 255), -1)


def process_video_colab(
    input_video,
    output_video="/content/cars_tracked_output.mp4",
    model_path="yolov8n.pt",
    conf=0.1,
    line_y=1600,
):
    if not os.path.exists(input_video):
        raise FileNotFoundError(f"Input video not found: {input_video}")

    model = YOLO(model_path)

    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open input video: {input_video}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    line_y = max(0, min(line_y, height - 1))

    temp_output = "/content/cars_tracked_temp.mp4"
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(temp_output, fourcc, fps, (width, height))

    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create output video: {temp_output}")

    unique_car_ids = set()
    last_centers = {}
    counted_ids_in = set()
    counted_ids_out = set()

    frame_index = 0
    written_frames = 0

    print("Processing started...")
    print("Input video:", input_video)
    print("Output video:", output_video)
    print("Total frames:", total_frames)
    print("FPS:", fps)
    print("Counting line y:", line_y)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_index += 1
        annotated_frame = frame.copy()

        draw_counting_line(annotated_frame, line_y, width)

        results = model.track(
            source=frame,
            persist=True,
            classes=[2],
            conf=conf,
            tracker="bytetrack.yaml",
            verbose=False
        )

        if results and len(results) > 0:
            result = results[0]

            if result.boxes is not None and len(result.boxes) > 0:
                boxes_xyxy = result.boxes.xyxy.cpu().numpy()
                confidences = result.boxes.conf.cpu().numpy()

                if result.boxes.id is not None:
                    track_ids = result.boxes.id.int().cpu().tolist()
                else:
                    track_ids = [-1] * len(boxes_xyxy)

                for box, score, track_id in zip(boxes_xyxy, confidences, track_ids):
                    x1, y1, x2, y2 = map(int, box)

                    if track_id != -1:
                        unique_car_ids.add(track_id)

                    cx = int((x1 + x2) / 2)
                    cy = int((y1 + y2) / 2)

                    if track_id != -1:
                        if track_id in last_centers:
                            _, prev_cy = last_centers[track_id]

                            if prev_cy < line_y <= cy and track_id not in counted_ids_in:
                                counted_ids_in.add(track_id)
                            elif prev_cy > line_y >= cy and track_id not in counted_ids_out:
                                counted_ids_out.add(track_id)

                        last_centers[track_id] = (cx, cy)

                    direction = ""
                    if track_id in counted_ids_in:
                        direction = " IN"
                    elif track_id in counted_ids_out:
                        direction = " OUT"

                    label = f"Car ID:{track_id} {score:.2f}{direction}"
                    draw_box_with_label(annotated_frame, x1, y1, x2, y2, label)

                    if track_id != -1:
                        cv2.circle(annotated_frame, (cx, cy), 7, (0, 0, 255), -1)

        panel_width = 1100
        panel_height = 220
        panel_x1 = (width - panel_width) // 2
        panel_y1 = 350
        panel_x2 = panel_x1 + panel_width
        panel_y2 = panel_y1 + panel_height

        cv2.rectangle(annotated_frame, (panel_x1, panel_y1), (panel_x2, panel_y2), (35, 35, 35), -1)
        cv2.rectangle(annotated_frame, (panel_x1, panel_y1), (panel_x2, panel_y2), (0, 255, 255), 4)

        stats_font_scale = 2.2
        stats_thickness = 6

        in_text = f"Cars going IN: {len(counted_ids_in)}"
        out_text = f"Cars going OUT: {len(counted_ids_out)}"

        (in_w, _), _ = cv2.getTextSize(in_text, cv2.FONT_HERSHEY_SIMPLEX, stats_font_scale, stats_thickness)
        (out_w, _), _ = cv2.getTextSize(out_text, cv2.FONT_HERSHEY_SIMPLEX, stats_font_scale, stats_thickness)

        in_x = panel_x1 + (panel_width - in_w) // 2
        out_x = panel_x1 + (panel_width - out_w) // 2

        in_y = panel_y1 + 80
        out_y = panel_y1 + 165

        cv2.putText(
            annotated_frame,
            in_text,
            (in_x, in_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            stats_font_scale,
            (0, 255, 0),
            stats_thickness,
            cv2.LINE_AA
        )

        cv2.putText(
            annotated_frame,
            out_text,
            (out_x, out_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            stats_font_scale,
            (0, 165, 255),
            stats_thickness,
            cv2.LINE_AA
        )

        writer.write(annotated_frame)
        written_frames += 1

        if frame_index % 30 == 0:
            print(f"Processed {frame_index}/{total_frames} frames")

    cap.release()
    writer.release()

    # Re-encode once for smoother playback, still saved in current Colab session
    os.system(
        f'ffmpeg -y -i "{temp_output}" -c:v libx264 -preset medium -crf 18 -pix_fmt yuv420p "{output_video}"'
    )

    print("\nDone.")
    print("Frames processed:", frame_index)
    print("Frames written:", written_frames)
    print("Saved as:", output_video)
    print("Total unique cars detected:", len(unique_car_ids))
    print("Cars going IN:", len(counted_ids_in))
    print("Cars going OUT:", len(counted_ids_out))

    return output_video


output_path = process_video_colab(
    input_video="/content/drive/MyDrive/fixed_input.mp4",
    output_video="/content/cars_tracked_output.mp4",
    model_path="yolov8n.pt",
    conf=0.1,
    line_y=1600
)

print("Final output saved at:", output_path)

Processing started...
Input video: /content/drive/MyDrive/fixed_input.mp4
Output video: /content/cars_tracked_output.mp4
Total frames: 372
FPS: 25.0
Counting line y: 1600
Processed 30/372 frames
Processed 60/372 frames
Processed 90/372 frames
Processed 120/372 frames
Processed 150/372 frames
Processed 180/372 frames
Processed 210/372 frames
Processed 240/372 frames
Processed 270/372 frames
Processed 300/372 frames
Processed 330/372 frames
Processed 360/372 frames

Done.
Frames processed: 372
Frames written: 372
Saved as: /content/cars_tracked_output.mp4
Total unique cars detected: 46
Cars going IN: 16
Cars going OUT: 13
Final output saved at: /content/cars_tracked_output.mp4


In [17]:
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>